In [3]:
import yfinance as yf
import time
import warnings
warnings.filterwarnings('ignore')

In [6]:
START = '1995-01-01'
results = {}

def fetch_yf(name, ticker, delay=2):
    time.sleep(delay)
    try:
        d = yf.download(ticker, start=START, progress=False, auto_adjust=True)
        d = d.dropna(how='all')
        if len(d) > 0:
            results[name] = {
                'ticker': ticker,
                'start':  str(d.index[0].date()),
                'end':    str(d.index[-1].date()),
                'rows':   len(d),
                'status': 'OK'
            }
            print(f"{name:10s} {ticker:15s}: {d.index[0].date()} -> {d.index[-1].date()} ({len(d):,} rows)")
        else:
            results[name] = {'ticker': ticker, 'status': 'NO DATA'}
            print(f"{name:10s} {ticker:15s}: NO DATA")
    except Exception as e:
        results[name] = {'ticker': ticker, 'status': 'ERROR'}
        print(f"{name:10s} {ticker:15s}: ERROR")

# equity 
print('\n=== EQUITY INDICES ===')
fetch_yf('SPX',   '^GSPC')
fetch_yf('STOXX', '^STOXX50E')
fetch_yf('FTSE',  '^FTSE')
fetch_yf('N225',  '^N225')

# Vol
print('\n=== VOL INDICES ===')
fetch_yf('VIX',   '^VIX')
fetch_yf('MOVE',  '^MOVE')
fetch_yf('EUVIX', '^EUVIX')

# commodity
print('\n=== COMMODITIES ===')
fetch_yf('GOLD',  'GC=F')


=== EQUITY INDICES ===
SPX        ^GSPC          : 1995-01-03 -> 2026-05-15 (7,895 rows)
STOXX      ^STOXX50E      : 2007-03-30 -> 2026-05-15 (4,792 rows)
FTSE       ^FTSE          : 1995-01-03 -> 2026-05-15 (7,923 rows)
N225       ^N225          : 1995-01-04 -> 2026-05-14 (7,689 rows)

=== VOL INDICES ===
VIX        ^VIX           : 1995-01-03 -> 2026-05-15 (7,895 rows)
MOVE       ^MOVE          : 2002-11-12 -> 2026-05-14 (5,811 rows)
EUVIX      ^EUVIX         : 2020-05-15 -> 2020-05-15 (1 rows)

=== COMMODITIES ===
GOLD       GC=F           : 2000-08-30 -> 2026-05-15 (6,451 rows)


In [22]:
def fetch_fred(name, series, delay=1):
    time.sleep(delay)
    try:
        url = (
            f'https://api.stlouisfed.org/fred/series/observations'
            f'?series_id={series}'
            f'&api_key={FRED_API_KEY}'
            f'&file_type=json'
            f'&observation_start=1990-01-01'
        )
        r = requests.get(url, timeout=15)
        data = r.json()
        if 'error_code' in data:
            print(f"{name:20s} {series:25s}: API ERROR — {data.get('error_message')}")
            return
        obs = [o for o in data['observations'] if o['value'] != '.']
        if obs:
            first = obs[0]['date']
            last  = obs[-1]['date']
            results[f'FRED_{name}'] = {
                'series': series,
                'start':  first,
                'end':    last,
                'rows':   len(obs),
                'status': 'OK'
            }
            print(f"{name:20s} {series:25s}: {first} -> {last} ({len(obs):,} rows)")
        else:
            print(f"{name:20s} {series:25s}: NO DATA")
    except Exception as e:
        print(f"{name:20s} {series:25s}: ERROR — {str(e)[:40]}")

print('=== TARGET PAIRS ===')
fetch_fred('EURUSD',    'DEXUSEU')
fetch_fred('GBPUSD',    'DEXUSUK')
fetch_fred('JPYUSD',    'DEXJPUS')

print('\n=== POLICY RATES ===')
fetch_fred('FED_FUNDS', 'FEDFUNDS')   # monthly native — forward-fill in pipeline

print('\n=== US YIELDS ===')
fetch_fred('US_10Y',    'DGS10')
fetch_fred('US_2Y',     'DGS2')
fetch_fred('DTB3',      'DTB3')

print('\n=== FOREIGN YIELDS MONTHLY ===')
fetch_fred('BUND_M',    'IRLTLT01DEM156N')
fetch_fred('GILT_M',    'IRLTLT01GBM156N')
fetch_fred('BTP_M',     'IRLTLT01ITM156N')
fetch_fred('JGB_M',     'IRLTLT01JPM156N')

print('\n=== VOL ===')
fetch_fred('VIX',       'VIXCLS')
fetch_fred('EVZ',       'EVZCLS')

print('\n=== US CREDIT SPREADS ===')
fetch_fred('US_HY',     'BAMLH0A0HYM2')
fetch_fred('US_IG',     'BAMLC0A0CM')

print('\n=== EUROPEAN CREDIT SPREADS ===')
fetch_fred('EU_HY',     'BAMLHE00EHY0EIM')
fetch_fred('EU_IG',     'BAMLEC0EEY1T10Y')

print('\n=== FUNDING STRESS ===')
fetch_fred('TED',       'TEDRATE')
fetch_fred('SOFR',      'SOFR')

print('\n=== OIL ===')
fetch_fred('WTI',       'DCOILWTICO')

=== TARGET PAIRS ===
EURUSD               DEXUSEU                  : 1999-01-04 -> 2026-05-08 (6,859 rows)
GBPUSD               DEXUSUK                  : 1990-01-02 -> 2026-05-08 (9,123 rows)
JPYUSD               DEXJPUS                  : 1990-01-02 -> 2026-05-08 (9,123 rows)

=== POLICY RATES ===
FED_FUNDS            FEDFUNDS                 : 1990-01-01 -> 2026-04-01 (436 rows)

=== US YIELDS ===
US_10Y               DGS10                    : 1990-01-02 -> 2026-05-13 (9,097 rows)
US_2Y                DGS2                     : 1990-01-02 -> 2026-05-13 (9,097 rows)
DTB3                 DTB3                     : 1990-01-02 -> 2026-05-13 (9,097 rows)

=== FOREIGN YIELDS MONTHLY ===
BUND_M               IRLTLT01DEM156N          : 1990-01-01 -> 2026-03-01 (435 rows)
GILT_M               IRLTLT01GBM156N          : 1990-01-01 -> 2026-03-01 (435 rows)
BTP_M                IRLTLT01ITM156N          : 1991-03-01 -> 2026-02-01 (420 rows)
JGB_M                IRLTLT01JPM156N          : 1990-0

In [21]:
def fetch_ecb(name, series_key, delay=2):
    time.sleep(delay)
    try:
        url = (
            f'https://data-api.ecb.europa.eu/service/data/'
            f'{series_key}'
            f'?startPeriod=1999-01-01&format=csvdata&detail=dataonly'
        )
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            print(f"{name:20s}: HTTP {r.status_code}")
            return

        # split on newlines, strip carriage returns
        lines = [l.strip() for l in r.text.strip().split('\n')]

        # first line is header — find TIME_PERIOD column index
        header = lines[0].split(',')
        try:
            date_idx  = header.index('TIME_PERIOD')
            value_idx = header.index('OBS_VALUE')
        except ValueError:
            print(f"{name:20s}: unexpected header — {header[:5]}")
            return

        # extract data rows
        data_lines = []
        for l in lines[1:]:
            if not l:
                continue
            parts = l.split(',')
            if len(parts) > value_idx and parts[value_idx].strip():
                data_lines.append((parts[date_idx], parts[value_idx]))

        if data_lines:
            first = data_lines[0][0]
            last  = data_lines[-1][0]
            print(f"{name:20s}: {first} -> {last} ({len(data_lines):,} rows)")
        else:
            print(f"{name:20s}: NO DATA ROWS after parsing")
    except Exception as e:
        print(f"{name:20s}: ERROR — {e}")

fetch_ecb('ECB_RATE',    'FM/B.U2.EUR.4F.KR.DFR.LEV')
fetch_ecb('BUND_AAA_10Y','YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y')
fetch_ecb('EA_ALL_10Y',  'YC/B.U2.EUR.4F.G_N_C.SV_C_YM.SR_10Y')

ECB_RATE            : 1999-01-01 -> 2025-06-11 (67 rows)
BUND_AAA_10Y        : 2004-09-06 -> 2026-05-14 (5,543 rows)
EA_ALL_10Y          : 2004-09-06 -> 2026-05-14 (5,543 rows)
